# Ordered Logistic Regression Results: Kenya Rangeland Management - Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All RecordSets, Fields, and Columns are referenced by their unique `@id` as per the FAIR data principles.

### Dataset Source
The `Croissant` schema for the dataset is available at:

    https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure 'mlcroissant' library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Authors: {[author for author in (metadata.author or [])]}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview

List all available record sets, fields, and their `@id`s from the Croissant metadata schema. This helps in selecting which elements to load for further analysis.

In [ ]:
# Explore available RecordSets and their fields, using their @id
if not dataset.record_sets:
    print("No explicit record sets are defined in the Croissant metadata.\n")
else:
    print("Available RecordSets and their fields:")
    for rset in dataset.record_sets:
        print(f"- RecordSet @id: {rset.id}")
        if hasattr(rset, 'fields') and rset.fields:
            for field in rset.fields:
                print(f"    - Field @id: {field.id} ({field.name})")
        else:
            print("    No fields found in this RecordSet.")

# For this dataset, if .record_sets is empty, we'll enumerate table-like resources from distribution entries
if not dataset.record_sets or len(dataset.record_sets) == 0:
    print("Attempting to enumerate available distributions as pseudo-record sets:")
    if hasattr(metadata, 'distribution'):
        for d in metadata.distribution:
            print(f"- Distribution @id: {d['@id']}")
    else:
        print("No distributions found in metadata.")

## 3. Data Extraction

Extract data from a specific record set (or data file) into a pandas DataFrame for analysis, using the record set and field `@id`s found in the previous step.

**Note:** In this FAIR² dataset, the record sets are presented as distributions (`@id`s):

- `http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8336ac61-9308-403f-8df3-28e120cc98f3`
- `http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8e507442-660d-4cfe-b2d9-f805d7abe725`

In [ ]:
# List record set (distribution) IDs based on previous overview
record_sets = [
    'http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8336ac61-9308-403f-8df3-28e120cc98f3',
    'http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8e507442-660d-4cfe-b2d9-f805d7abe725'
]

dataframes = {}

for rs_id in record_sets:
    try:
        # Load as Croissant record set by @id
        df_iter = dataset.records(record_set=rs_id)
        records = list(df_iter)
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for RecordSet @id: {rs_id}, shape={dataframes[rs_id].shape}")
        else:
            print(f"No records found for RecordSet @id: {rs_id}")
    except Exception as e:
        print(f"Error loading RecordSet @id {rs_id}: {e}")

# Show columns for the first loaded DataFrame
if dataframes:
    first_id = next(iter(dataframes.keys()))
    print(f"\nColumns in first RecordSet (@id: {first_id}):")
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())
else:
    print("No dataframes loaded; the dataset may require additional access permissions or metadata.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Below, we select a numeric column by its Croissant field `@id` (or, if field IDs unavailable, by column name), filter for values above a threshold, normalize the column, and group by a categorical `@id` where possible.

*Be sure to substitute `<field_id>` or column names with those you see printed above.*

In [ ]:
# Select the main DataFrame (use first loaded RecordSet)
if dataframes:
    record_set_id = first_id
    df = dataframes[record_set_id]
    # Example: pick a likely numeric field (replace with observed field/column name if possible)
    possible_numeric_fields = [col for col in df.columns if (df[col].dtype in ['float64', 'int64'])]

    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]  # Use first numeric field
        print(f"Using numeric field for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'O' else 10

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (count={len(filtered_df)}):")
        display(filtered_df.head())

        normcol = f"{numeric_field}_normalized"
        filtered_df[normcol] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, normcol]].head())

        # Group by a categorical field, if available
        possible_categorical = [col for col in df.columns if df[col].dtype == 'O' and col != numeric_field]
        if possible_categorical:
            group_field = possible_categorical[0]
            print(f"\nGrouping by: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped.head())
        else:
            print("No categorical fields available to group by.")
    else:
        print("No numeric fields found in the DataFrame.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. This example plots the distribution of a numeric field and, if applicable, a boxplot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group, if grouping was possible
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"'{numeric_field}' by '{group_field}'")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion

In this notebook, we have demonstrated how to explore a FAIR dataset described by a Croissant schema using the `mlcroissant` library.

We loaded dataset metadata, discovered available record sets and their fields using their `@id`s, extracted the data, and performed basic exploratory analysis including filtering, normalization, grouping, and visualization.

For more advanced analyses, refer to specific field or column `@id`s as shown in the schema, and consult the [mlcroissant documentation](https://github.com/mlcommons/croissant) for further programmatic access to FAIR data.